# QA System Training Notebook for Google Colab

This notebook trains all models for the QA system and saves them to Google Drive.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted successfully!")

In [ ]:
# Install dependencies
!pip install pandas numpy scikit-learn nltk gensim sentence-transformers streamlit joblib pyngrok

In [ ]:
# Setup paths
import os
import sys
from pathlib import Path

# Create directories
os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/models', exist_ok=True)
os.makedirs('/content/src', exist_ok=True)

print("Directories created successfully!")

In [ ]:
# Copy dataset from Drive (assuming it's in /content/drive/My Drive/data/)
!cp '/content/drive/My Drive/data/train.csv' '/content/data/train.csv' 2>/dev/null || echo "Train file not found in Drive"
!cp '/content/drive/My Drive/data/val.csv' '/content/data/val.csv' 2>/dev/null || echo "Val file not found in Drive"
!cp '/content/drive/My Drive/data/test.csv' '/content/data/test.csv' 2>/dev/null || echo "Test file not found in Drive"

# Verify data
import os
data_files = os.listdir('/content/data')
print(f"Data files available: {data_files}")

In [ ]:
# Add source code path
sys.path.insert(0, '/content')

# Import necessary modules
import pandas as pd
import numpy as np
import joblib
from src.utils import load_data, create_feature_text
from src.preprocess import preprocess_and_save
from src.model_a_supervised import train_and_evaluate_supervised
from src.model_a_unsupervised import train_and_evaluate_clustering
from src.model_b_distractors import evaluate_distractors
from src.model_b_hints import evaluate_hints

print("All modules imported successfully!")

In [ ]:
# Load data
train_df, val_df, test_df = load_data('/content/data')
print(f"Data loaded. Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

In [ ]:
# Preprocess data
X_train, X_val, X_test, y_train, y_val, y_test = preprocess_and_save(
    train_df, val_df, test_df, '/content/models'
)

# Load vectorizer and label encoder for later use
tfidf_vectorizer = joblib.load('/content/models/tfidf_vectorizer.pkl')
label_encoder = joblib.load('/content/models/label_encoder.pkl')

print("Preprocessing completed!")

In [ ]:
# Train and evaluate supervised models
supervised_metrics = train_and_evaluate_supervised(
    X_train, y_train, X_val, y_val, '/content/models'
)

print("\nSupervised models trained and evaluated!")

In [ ]:
# Train and evaluate clustering
clustering_metrics = train_and_evaluate_clustering(
    X_train, X_val, y_val, tfidf_vectorizer, '/content/models'
)

print("\nClustering model trained and evaluated!")

In [ ]:
# Evaluate distractors
distractors_metrics = evaluate_distractors(test_df, tfidf_vectorizer)

print("\nDistracts evaluated!")

In [ ]:
# Evaluate hints
hints_metrics = evaluate_hints(test_df, tfidf_vectorizer)

print("\nHints evaluated!")

In [ ]:
# Aggregate all metrics
all_metrics = {
    'supervised': supervised_metrics.to_dict('records'),
    'clustering': clustering_metrics,
    'distractors': distractors_metrics,
    'hints': hints_metrics
}

# Save metrics to JSON
import json
metrics_path = '/content/models/metrics.json'
with open(metrics_path, 'w') as f:
    json.dump(all_metrics, f, indent=2)

print(f"Metrics saved to {metrics_path}")

In [ ]:
# Copy models to Google Drive
import shutil

drive_models_path = '/content/drive/My Drive/models'
os.makedirs(drive_models_path, exist_ok=True)

# Copy all model files
for file in os.listdir('/content/models'):
    src = os.path.join('/content/models', file)
    dst = os.path.join(drive_models_path, file)
    if os.path.isfile(src):
        shutil.copy(src, dst)
        print(f"Copied {file} to Drive")

print(f"\nAll models saved to {drive_models_path}")

In [ ]:
# Display final summary
print("\n" + "="*60)
print("TRAINING COMPLETE - SUMMARY")
print("="*60)
print(f"\nModels saved to: /content/models/")
print(f"Drive backup: {drive_models_path}")
print(f"\nAvailable models:")
for file in os.listdir('/content/models'):
    print(f"  - {file}")
print("="*60)